## B01 · 1. Meet ten customers

A phone company cannot call everyone. We want to find people who may leave. Save a copy: **<u>File → Save a copy in Drive</u>**. Click ▶ to the left of each code cell to run it. Run the next two cells. These are prepared examples; no files are needed. To write an answer, double-click its text cell, replace the dots with your words, then press Shift+Enter.

Look for eight correct answers. **Question:** Did we find either person who left?

In [ ]:
customers = ['C%02d' % i for i in range(1,11)]
scores = [80,40,60,35,30,20,15,10,5,1]
truth = [1,1,0,0,0,0,0,0,0,0]  # 1 = Left
for c,s,y in zip(customers,scores,truth):
    print(c, 'Left' if y else 'Stayed', '| prepared score:', s)

In [ ]:
print('Correct:',sum(y == 0 for y in truth),'of 10 | leavers found: 0 of 2')

## B02 · 2. Count people, then read the scores

Each prepared score is from 0 to 100. A higher score means the model expects that customer is more likely to leave. These toy scores are already given. For the same ten customers, use a new rule. The previous rule said everyone stays. Now call anyone with a score of at least 50. First count the two people who left. We found one: **<u>Recall</u>** is 1/2. Next count the two people called. One call was right: **<u>Precision</u>** is 1/2. Accuracy is 8/10. F1 combines precision and recall; here it is 0.5. Run the cell to check.

**Question:** Which two people were called?

**My answer:** … **TODO 1/3:** Complete Precision, Recall and F1; return all four scores.

<details><summary>Reference: metric formulas</summary>TP = leavers called; FP = people called who stayed; FN = leavers missed. Precision = TP / (TP + FP). Recall = TP / (TP + FN). F1 = 2 × Precision × Recall / (Precision + Recall). Use 0 when a denominator is 0.</details>

In [ ]:
def metric_scores(truth, predicted):
    pairs = list(zip(truth, predicted))
    tp = sum(a == 1 and b == 1 for a, b in pairs)
    fp = sum(a == 0 and b == 1 for a, b in pairs)
    fn = sum(a == 1 and b == 0 for a, b in pairs)
    tn = sum(a == 0 and b == 0 for a, b in pairs)
    accuracy = (tp + tn) / len(truth)
    # TODO 1/3: calculate Precision, Recall and F1; use 0 for an empty group.
    # Return all four named values in a dictionary.
    raise NotImplementedError('Complete Precision, Recall and F1')

toy_at_50=[int(s>=50) for s in scores]
result=metric_scores(truth,toy_at_50)
assert all(abs(result[k]-v)<1e-9 for k,v in {'Accuracy':.8,'Precision':.5,'Recall':.5,'F1':.5}.items())
print(result)
unequal=metric_scores([1,1,0],[1,1,1])
assert abs(unequal['Precision']-2/3)<1e-9 and abs(unequal['Recall']-1)<1e-9 and abs(unequal['F1']-.8)<1e-9

## B03 · 3. Open the real customer data

Now a model will learn from old records. Run the folded setup and upload `telco_source.csv` and `M3C5_Runtime.zip`. Training teaches the model. Validation means checking our choices on other customers. The final test stays closed. In old training records, `Churn` tells us whether each customer left. It is the known answer used to teach the model, not a customer input at prediction time. Validation answers are used only to check the model. A future customer’s answer is not known yet.

**Question:** Which records teach the model?

In [ ]:
#@title ▶ Prepare the frozen lesson data
from pathlib import Path
from zipfile import ZipFile, BadZipFile
import sys
if 'get_ipython' in globals():
    get_ipython().run_line_magic('matplotlib', 'inline')
try:
    from imblearn.over_sampling import RandomOverSampler, SMOTE
except ImportError:
    try:
        import google.colab
    except ImportError:
        raise ImportError('Local environment needs imbalanced-learn. Ask the mentor or use Colab.') from None
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imbalanced-learn==0.14.2'])
required = ['telco_source.csv', 'M3C5_Runtime.zip']
if any(not Path(name).exists() for name in required):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place the issued CSV and M3C5_Runtime.zip beside this notebook; rerun B03.') from None
    print('Select BOTH telco_source.csv and M3C5_Runtime.zip from the lesson package.')
    files.upload()
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError('Missing: ' + ', '.join(missing) + '. Keep the original file names.')
folder = Path('_m3c5_runtime')
folder.mkdir(exist_ok=True)
try:
    with ZipFile('M3C5_Runtime.zip') as archive:
        for name in ['m3c5_runtime.py', 'm3c5_style.py', 'M3_Split_Card.json']:
            (folder / name).write_bytes(archive.read(name))
except (BadZipFile, KeyError):
    raise ValueError('Wrong or damaged runtime ZIP. Upload the issued ZIP again.') from None
if str(folder.resolve()) not in sys.path:
    sys.path.insert(0, str(folder.resolve()))
exec(compile((folder / 'm3c5_runtime.py').read_text(encoding='utf-8'), 'm3c5_runtime.py', 'exec'), globals())

feature_mode='mentor_fallback'  # optional: 'c4_personal'
personal_c4_columns=[]  # optional: exact encoded C4 names
_ = config()  # validate the fixed choice

## B04 · 4. Try our starting model

Baseline is our starting model. Run it to get a result for comparison. Read leavers found and unnecessary calls. V01 is a real validation customer.

**Question:** What happened to V01?

In [ ]:
baseline_row, baseline_probability = record_method('Baseline')

## B05 · 5. Change who gets a call

A cutoff is the score needed for a call. Run 50, 40 and 30 on the same scores, then check the real model. The toy scores use 0–100 and the real model uses 0–1: 50/100 = .5, 40/100 = .4, and 30/100 = .3. Equal scores count. The model does not learn again. We return to .5 for training comparisons.

**Question:** What changes when the cutoff falls? **TODO 2/3:** Return integer labels for scores at or above the cutoff.

In [ ]:
def threshold_labels(probability, threshold):
    # TODO 2/3: use np.asarray(probability) and .astype(int)
    raise NotImplementedError('Complete threshold_labels')

for t in [50,40,30]:
    labels=threshold_labels(scores,t)
    called=[c for c,p in zip(customers,labels) if p]
    found=sum(a==1 and b==1 for a,b in zip(truth,labels))
    extra=sum(a==0 and b==1 for a,b in zip(truth,labels))
    print('Cutoff',t,':',called,'| found',found,'of 2 | unnecessary calls',extra)
threshold_sweep()

## B06 · 6. Repeat rare training examples

ROS copies rare training rows. Run it and compare the result with Baseline. Copies are not new people. Validation rows stay unchanged.

**Question:** Which rows were copied?

In [ ]:
ros_row, ros_probability = record_method('RandomOverSampler')

## B07 · 7. Make in-between training examples

SMOTE makes new training examples between existing rows. For (10 months, bill 100) and (20 months, bill 200), one example is (15, 150). It is not a real customer. A value between 0 and 1 needs care for a Yes/No detail. Run SMOTE.

**Question:** How is this different from copying?

In [ ]:
smote_row, smote_probability = record_method('SMOTE')

## B08 · 8. Change error weights and compare

Few customers have the rare outcome Left. Weights give mistakes on these customers more importance during learning. They add no rows. Run the model and compare all four on the same validation customers at .5. Baseline found 204 leavers and made 120 unnecessary calls. ROS found 302 and made 311 unnecessary calls. ROS found more leavers, but made more unnecessary calls.

**Question:** Compared with Baseline, how many leavers did ROS find, and how many unnecessary calls did each make?

**My answer:** …

In [ ]:
weight_row, weight_probability = record_method('class_weight')
compare_methods()

## B09 · 9. Check that we did not peek

We compared methods. Now check whether preparation used information from checking customers. Looking at checking data too early is called leakage. A setting is a number learned from data, such as an average bill.

Before a call, can we know why a customer will later cancel? No. Use only details known before the call.

**Mean example**

Training bills 100 and 200 have an average of 150. The checking bill 300 is 300 − 150 = 150 above that usual bill. One preparation step can use this difference as a number for the model. This is one part of preparation, not the full scaler output. The average is a learned setting, not a model score. If we include the checking bill, the average wrongly becomes 200.

**Why this example**

The four methods above already use training data only for preparation. We deliberately made the wrong example below to show what leakage looks like. This check does not change those four models or resample their data. The score may stay the same; the rule still needs the fix.

**Run these 3 cells**

1. Click ▶ next to the first cell. Find the line marked WRONG. Do not edit it.
2. Complete TODO 3, then run the function cell.
3. Click ▶ next to the third cell. Find the correct row count: 4,225, not 5,634.

**Question:** Which rows may teach the settings?

**My answer:** …

<details><summary>How to read the code</summary>`train_numeric` holds training customer data; `validation_numeric` holds checking customer data. `pd.concat` combines the two groups, and `fit` learns numbers from them.</details>

<details><summary>More support: groups and time</summary>Keep records from the same person together when checking new people. For future predictions, train on earlier records and check later records. Stratification keeps class shares similar but cannot do either job by itself.</details> **TODO 3/3:** Fit the scaler on training rows only and return it.

In [ ]:
train_numeric, validation_numeric = leakage_inputs()
wrong_scaler = StandardScaler().fit(pd.concat([train_numeric, validation_numeric]))  # WRONG
print('Wrong: learned from', wrong_scaler.n_samples_seen_, 'customers. Training only:', len(train_numeric))


In [ ]:
def fit_train_scaler(train_numeric, validation_numeric):
    scaler = StandardScaler()
    # TODO 3/3: fit scaler on train_numeric only and return it.
    raise NotImplementedError('Complete train-only fit')


In [ ]:
compare_scalers()

## B10 · 10. Choose and save

The team can make at most **450 calls**. Calls = leavers found + unnecessary calls. Among measured options within the limit, find more leavers. Baseline .4 found 253, made 185 unnecessary calls, and missed 121: 438 calls total. Baseline .5 found 204 and made 120 unnecessary calls: 324 total. This limit is a teaching assumption.

Choose a measured option and run the cell. The two Baseline .5 entries are the same model and cutoff. **Question:** Why does your choice fit the limit?

**My reason:** …

**Class ends here. Save your notebook.** The final test stays closed.

In [ ]:
#@title Choose a measured option
choice = 'Baseline threshold | 0.4' #@param ['Baseline | 0.5', 'RandomOverSampler | 0.5', 'SMOTE | 0.5', 'class_weight | 0.5', 'Baseline threshold | 0.3', 'Baseline threshold | 0.4', 'Baseline threshold | 0.5']
chosen_method, chosen_threshold_text = choice.rsplit(' | ', 1)
chosen_threshold = float(chosen_threshold_text)
fresh_facts()

## H01 · Homework 1. Explain your choice

Write a roughly one-page recommendation below. Use your observed numbers.

- Say which customer details were used and explain the imbalance.
- Compare all four methods and their measured results. Explain your cutoff and tradeoff.
- Explain the wrong fit and repair. Say why preparation and sampling use training data only.
- Keep the final test closed. Describe one limitation and the next check.

In [ ]:
recommendation = """

"""
recommendation_choice=(chosen_method,chosen_threshold)
recommendation_feature_mode=feature_mode

## H02 · Homework 2. Export your results

Read your answers and recommendation. Change the acknowledgement to True only after manual review. Then run the export.

In [ ]:
written_notes_reviewed=False  # change after manual review
export_current()

## H03 · Homework 3. Save, reopen and submit

Save and download the executed notebook. Download `M3C5_Comparison.csv` and `M3C5_Recommendation.md`. Reopen all three files and check they match. Submit the three files through the existing LMS route.